# Instruction Adherence QA Validation

## 1. Purpose

Instruction Adherence checks whether the output follows explicit constraints such as counts, required structure, prohibitions, and source restrictions.


## 2. Imports and output location

The helper locates the repository root whether Jupyter starts at the repository root or inside `notebooks/qa`.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from idp_eval import EvaluationCase, EvaluationFramework, create_azure_judge
from idp_eval.judges import AzureJudgeConfig

from idp_eval import InstructionAdherenceEvaluator


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "idp_eval").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the idp-eval repository.")


REPO_ROOT = find_repo_root()
OUTPUT_DIR = REPO_ROOT / "qa_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Configure the judge

Replace every placeholder before running. No Phoenix server is required. If your
application already constructs a compatible judge, you may replace this cell
with that existing construction. Keep credentials in your application's secret
management system rather than saving them in this notebook.


In [ ]:
azure_config = AzureJudgeConfig(
    model="YOUR_AZURE_DEPLOYMENT",
    azure_endpoint="YOUR_AZURE_ENDPOINT",
    tenant_id="YOUR_TENANT_ID",
    client_id="YOUR_CLIENT_ID",
    client_secret="YOUR_CLIENT_SECRET",
    api_version="2024-12-01-preview",
    timeout=180,
    proxy_url=None,
    verify_ssl=True,
    reasoning_effort=None,
)

judge = create_azure_judge(config=azure_config)


## 4. Five mock evaluation cases

These cases intentionally span clear pass, partial, and fail behaviors. `expected_behavior` is a human QA aid, not an exact model-score assertion.


In [ ]:
cases = [
    EvaluationCase(
        case_id="INST-001",
        input="Recommend approved service options.",
        context="Approved options: Standard, Plus, Premium.",
        instructions="""Return exactly 3 recommendations.
Every recommendation must contain a title and description.
Do not include implementation details.
Only use approved options from the supplied context.""",
        output=[
            {"title": "Standard", "description": "Core service capabilities."},
            {"title": "Plus", "description": "Expanded service capabilities."},
            {"title": "Premium", "description": "Complete service capabilities."},
        ],
    ),
    EvaluationCase(
        case_id="INST-002",
        instructions="Return exactly 3 recommendations.",
        output=["Recommendation A", "Recommendation B"],
    ),
    EvaluationCase(
        case_id="INST-003",
        instructions="Every item must contain both a title and a description.",
        output=[
            {"title": "Option A", "description": "First option."},
            {"title": "Option B"},
        ],
    ),
    EvaluationCase(
        case_id="INST-004",
        instructions="Provide a business summary. Do not include implementation details.",
        output="The workflow reduces processing delays. Implement it with Python workers and a PostgreSQL queue.",
    ),
    EvaluationCase(
        case_id="INST-005",
        context="Approved channels: email and web portal.",
        instructions="""Return exactly 2 options.
Give every option a title.
Only use approved channels.
Do not include pricing.""",
        output=[
            {"title": "Email", "pricing": "$10 per month"},
            {"title": "Web portal"},
        ],
    ),
]

expected_behavior = {
    "INST-001": "all instructions followed",
    "INST-002": "exact-count instruction violated — 2 returned instead of 3",
    "INST-003": "required structure violated — one description missing",
    "INST-004": "explicit prohibition violated — implementation details included",
    "INST-005": "one violation — pricing included; remaining instructions followed",
}


## 5. Inspect the mock inputs


In [ ]:
case_rows = []
for case in cases:
    case_rows.append({
        "case_id": case.case_id,
        "expected_behavior": expected_behavior[case.case_id],
        "context": getattr(case, "context"),
        "instructions": getattr(case, "instructions"),
        "output": getattr(case, "output"),
    })

cases_df = pd.DataFrame(case_rows)
display(cases_df)


## 6. Configure one evaluator and Excel output

This notebook runs exactly one metric. `resume=False` creates a fresh QA workbook and no Phoenix tracing is configured.


In [ ]:
excel_path = OUTPUT_DIR / "instruction_adherence_validation.xlsx"
evaluator = InstructionAdherenceEvaluator(verbose=True)
framework = EvaluationFramework(
    evaluators=[evaluator],
    judge=judge,
    output="excel",
    excel_path=str(excel_path),
    resume=False,
)

results = framework.evaluate_many(
    cases,
    run_name="qa-validation",
    dataset_name="mock-acceptance-cases",
    show_progress=True,
)


## 7. Result summary


In [ ]:
METRIC_NAME = "instruction_adherence"
summary_rows = []
for case, result_map in zip(cases, results, strict=True):
    result = result_map[METRIC_NAME]
    summary_rows.append({
        "case_id": case.case_id,
        "expected_behavior": expected_behavior[case.case_id],
        "score": result.score,
        "label": result.label,
        "explanation": result.explanation,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


## 8. Inspect Excel output

The workbook summary is in `evaluations`; item-level evidence is in `instruction_adherence_items`.


In [ ]:
evaluations_df = pd.read_excel(excel_path, sheet_name="evaluations")
display(evaluations_df)

details_df = pd.read_excel(excel_path, sheet_name="instruction_adherence_items")
display(details_df)


## 9. Sanity assertions

These assertions validate framework/output behavior and broad direction only; they do not require exact LLM-generated fractions.


In [ ]:
assert len(results) == 5
assert excel_path.exists()
assert all(METRIC_NAME in result_map for result_map in results)
assert len(evaluations_df) == 5
assert set(evaluations_df["key_id"]) == {case.case_id for case in cases}
assert len(details_df) >= 5
assert results[0][METRIC_NAME].score >= results[1][METRIC_NAME].score
assert results[0][METRIC_NAME].score >= results[3][METRIC_NAME].score
print("Instruction Adherence QA sanity checks passed.")


## 10. Close judge resources and report the workbook path


In [ ]:
judge.close()
print(f"Excel output: {excel_path.resolve()}")
